# Laxman AI Avatar Studio — MuseTalk v1.5

**Permanent GitHub notebook + private Google Drive media/models + disposable Colab GPU.**

Run the cells from top to bottom after a fresh Colab runtime. The notebook creates a reproducible environment, discovers private assets, creates jobs, runs MuseTalk v1.5, records results, and performs health checks.

## Architecture & privacy

- GitHub: code, notebook, configuration and UI only.
- Google Drive: avatars, original audio, MuseTalk/model weights, queues, logs and generated videos.
- Colab: temporary GPU worker.
- Voice cloning is disabled: the selected original recording is used directly as audio input.
- Never commit private media, model weights, credentials or tokens to GitHub.

In [ ]:
from google.colab import drive
from pathlib import Path
import json, os, sys, subprocess, shutil, time, uuid

drive.mount('/content/drive')
ROOT=Path('/content/drive/MyDrive/Laxman AI Avatar Studio')
FOLDERS=['avatars','voices','models','outputs','temp','logs','jobs/queued','jobs/processing','jobs/completed','jobs/failed']
for p in FOLDERS: (ROOT/p).mkdir(parents=True,exist_ok=True)
print('Drive root:',ROOT)
print('Folders ready:',len(FOLDERS))


In [ ]:
# PHASE 1–2 — RESTORE THE PERMANENT CODEBASE
%cd /content
if not Path('/content/AI-Avatar-Studio').exists():
    subprocess.run(['git','clone','--depth','1','https://github.com/LaxmanNepal/AI-Avatar-Studio.git','/content/AI-Avatar-Studio'],check=True)
else:
    subprocess.run(['git','-C','/content/AI-Avatar-Studio','pull','--ff-only'],check=False)
print('Repository ready:',Path('/content/AI-Avatar-Studio'))


In [ ]:
# PHASE 3–6 — REPRODUCIBLE MUSETalk ENVIRONMENT
subprocess.run(['python3.10','/content/AI-Avatar-Studio/backend/colab_bootstrap.py'],check=True)
print('Bootstrap completed.')


In [ ]:
# PHASE 7 — VERIFY GPU + PYTORCH + MMENGINE/MMCV
PY='/content/musetalk-env/bin/python'
check="import torch; print('Torch:',torch.__version__); print('CUDA:',torch.cuda.is_available()); print('GPU:',torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE'); import mmcv; print('MMCV:',mmcv.__version__)"
subprocess.run([PY,'-c',check],check=True)


In [ ]:
# PHASE 8 — REBUILD THE MODEL LINK AFTER RUNTIME RESET
models_link=Path('/content/MuseTalk/models')
drive_models=ROOT/'models'
if models_link.is_symlink() or models_link.exists():
    if models_link.is_symlink() or models_link.is_file(): models_link.unlink()
    else: shutil.rmtree(models_link)
models_link.symlink_to(drive_models,target_is_directory=True)
print('MuseTalk models ->',models_link.resolve())


In [ ]:
# PHASE 9 — MODEL + PRIVATE ASSET HEALTH CHECK
required=['models/musetalkV15/unet.pth','models/musetalkV15/musetalk.json','models/sd-vae/config.json','models/sd-vae/diffusion_pytorch_model.bin','models/whisper/config.json','models/whisper/pytorch_model.bin','models/whisper/preprocessor_config.json','models/dwpose/dw-ll_ucoco_384.pth','models/face-parse-bisent/79999_iter.pth','models/face-parse-bisent/resnet18-5c106cde.pth']
for rel in required:
    p=ROOT/rel; print(('OK  ' if p.exists() else 'MISS'),rel)
assert all((ROOT/r).exists() for r in required),'One or more required model files are missing.'
print('Models: all required files present.')


In [ ]:
# PHASE 10 — DISCOVER PRIVATE AVATARS + ORIGINAL VOICE
video_files=sorted((ROOT/'avatars').rglob('*.mp4'))
voice_files=sorted([*ROOT.glob('voices/*.wav'),*ROOT.glob('temp/*.wav')])
print('Avatar MP4s found:',len(video_files))
for p in video_files: print('  ',p.relative_to(ROOT))
print('Voice WAVs found:',len(voice_files))
for p in voice_files: print('  ',p.relative_to(ROOT))
test_avatar=ROOT/'temp/laxman_avatar_test.mp4'
test_voice=ROOT/'temp/laxman_voice_musetalk.wav'
assert test_avatar.exists(),'Missing temp/laxman_avatar_test.mp4'
assert test_voice.exists(),'Missing temp/laxman_voice_musetalk.wav'
print('Known test assets: READY')


In [ ]:
# PHASE 11 — CREATE A REAL TEST JOB
job_id='laxman-test-'+uuid.uuid4().hex[:8]
job={'schema_version':1,'id':job_id,'avatar_path':str(test_avatar),'audio_path':str(test_voice),'batch_size':4,'fps':24}
job_file=ROOT/'jobs/queued'/f'{job_id}.json'
job_file.write_text(json.dumps(job,indent=2),encoding='utf-8')
print('TEST JOB READY')
print(job_file)
print(json.dumps(job,indent=2))


In [ ]:
# PHASE 12 — RUN THE QUEUE
subprocess.run(['python','/content/AI-Avatar-Studio/backend/run_worker.py'],check=True)


In [ ]:
# PHASE 13 — INSPECT OUTPUT + RESULT METADATA
outs=sorted((ROOT/'outputs').glob('*.mp4'),key=lambda p:p.stat().st_mtime,reverse=True)
print('Generated MP4 files:',len(outs))
for p in outs[:10]: print(p.name,round(p.stat().st_size/1024/1024,2),'MB')
completed=sorted((ROOT/'jobs/completed').glob('*.json'),key=lambda p:p.stat().st_mtime,reverse=True)
failed=sorted((ROOT/'jobs/failed').glob('*.json'),key=lambda p:p.stat().st_mtime,reverse=True)
print('Completed metadata:',len(completed))
print('Failed metadata:',len(failed))
if failed: print(failed[0].read_text(encoding='utf-8'))


In [ ]:
# PHASE 14 — OPTIONAL: CREATE JOB FROM A PRIVATE AVATAR + ORIGINAL VOICE
def make_job(avatar_path, audio_path=test_voice, batch_size=4, fps=24, note=''):
    avatar_path=Path(avatar_path); audio_path=Path(audio_path)
    assert avatar_path.exists(),f'Missing avatar: {avatar_path}'
    assert audio_path.exists(),f'Missing audio: {audio_path}'
    jid='job-'+time.strftime('%Y%m%d-%H%M%S')+'-'+uuid.uuid4().hex[:6]
    job={'schema_version':1,'id':jid,'avatar_path':str(avatar_path),'audio_path':str(audio_path),'batch_size':int(batch_size),'fps':int(fps),'note':note}
    path=ROOT/'jobs/queued'/f'{jid}.json'; path.write_text(json.dumps(job,indent=2),encoding='utf-8')
    return path,job
print('Helper ready: make_job(...)')


## Phase 15 — Using the three avatar angles

The dashboard should reference the private Drive paths from `data/avatars.json`. The notebook can also use `make_job()` directly after you place the three source MP4s under `avatars/`.

Example:

```python
path, job = make_job(ROOT / 'avatars' / 'YOUR_AVATAR_FILE.mp4')
print(path)
```

Then run the **RUN THE QUEUE** cell.

In [ ]:
# PHASE 16 — FINAL HEALTH CHECK
checks={
  'Drive root':ROOT.exists(),
  'MuseTalk repo':Path('/content/MuseTalk').exists(),
  'Worker repo':Path('/content/AI-Avatar-Studio/backend/run_worker.py').exists(),
  'v1.5 UNet':(ROOT/'models/musetalkV15/unet.pth').exists(),
  'test avatar':test_avatar.exists(),
  'test voice':test_voice.exists(),
  'output folder':(ROOT/'outputs').exists(),
  'queue folders':all((ROOT/p).exists() for p in ['jobs/queued','jobs/processing','jobs/completed','jobs/failed'])
}
for k,v in checks.items(): print(('PASS ' if v else 'FAIL ')+k)
assert all(checks.values()),'Health check failed.'
print('Studio worker health check passed.')


## Daily workflow after setup

1. Start Colab with GPU.
2. Mount Drive.
3. Run the restore/bootstrap cells after a runtime reset.
4. Create a job from the dashboard or `make_job()`.
5. Run the queue worker.
6. Check `outputs/` and `jobs/completed/`.
7. If a job fails, inspect `jobs/failed/` and `logs/`.

**Important:** a successful bootstrap is not the same as a successful generation. The first real test must finish with an MP4 in `outputs/`.

## Job schema

```json
{
  "schema_version": 1,
  "id": "job-001",
  "avatar_path": "/content/drive/MyDrive/Laxman AI Avatar Studio/avatars/example.mp4",
  "audio_path": "/content/drive/MyDrive/Laxman AI Avatar Studio/temp/laxman_voice_musetalk.wav",
  "batch_size": 4,
  "fps": 24,
  "note": "optional"
}
```

Private media and model weights remain outside GitHub.

## Phase 19 — Continuous queue worker

After the environment and model checks pass, the worker can stay alive and watch `jobs/queued/`. Every new job JSON is processed automatically; failed jobs are recorded under `jobs/failed/` and the worker continues watching the queue.

**Start:** `python /content/AI-Avatar-Studio/backend/run_worker.py --loop --poll-seconds 10`

Stop the cell/runtime when you are finished. Colab remains disposable; Drive remains persistent.

In [ ]:
# PHASE 19 — START CONTINUOUS QUEUE WORKER
print('Watching:', ROOT/'jobs/queued')
print('Stop the cell/runtime to stop the worker.')
subprocess.run(['python','/content/AI-Avatar-Studio/backend/run_worker.py','--loop','--poll-seconds','10'],check=True)


## Phase 20 — Safe Drive status synchronization

The worker now generates a sanitized `sync/studio-status.json` manifest after successful and failed jobs. It contains only job state and non-sensitive media metadata.

**Never published:** Drive paths, local paths, credentials, raw error text, logs, source audio/avatar paths, or video bytes.

The GitHub Pages dashboard still does not access Google Drive directly. A future transport can publish this sanitized manifest over HTTPS after its security is configured.

In [ ]:
# PHASE 20 — BUILD SANITIZED STATUS MANIFEST
subprocess.run(['python','/content/AI-Avatar-Studio/backend/sync_status.py'],check=True)
status_file=ROOT/'sync/studio-status.json'
print(status_file)
print(status_file.read_text(encoding='utf-8'))


## Phase 21 — Deterministic job output isolation

Each queued job now gets its own temporary output directory under `outputs/_jobs/<job-id>`. The worker refuses ambiguous results, validates exactly one MP4, moves it to `outputs/<job-id>.mp4`, and removes the temporary directory on success or failure.

In [ ]:
# PHASE 21 — VERIFY WORKER HARDENING
worker=Path('/content/AI-Avatar-Studio/backend/run_worker.py')
text=worker.read_text(encoding='utf-8')
assert 'job_out=OUT/"_jobs"/run_id' in text
assert 'refusing ambiguous output selection' in text
print('Deterministic output isolation: READY')


## Phase 22 — Crash recovery + stale-job detection

Processing jobs now carry a worker ID and heartbeat. The worker refreshes the heartbeat while MuseTalk runs. If Colab crashes or disconnects and the heartbeat becomes older than the default 60-minute stale threshold, the next worker moves the job back to `jobs/queued/` for retry. Recovery is logged without exposing private media paths.

The processing status filename is also normalized from the job `id` so queue filename and JSON ID cannot leave inconsistent status metadata.

In [ ]:
# PHASE 22 — VERIFY RECOVERY HARDENING
worker=Path('/content/AI-Avatar-Studio/backend/run_worker.py')
text=worker.read_text(encoding='utf-8')
assert 'recover_stale_jobs' in text
assert 'heartbeat_at' in text
assert 'WORKER_ID' in text
assert 'STALE_AFTER_SECONDS=3600' in text
assert 'processing_meta=P/f"{run_id}.status.json"' in text
print('Crash recovery + heartbeat protection: READY')
